In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


name, memory.total [MiB]
Tesla T4, 15360 MiB


## Install Unsloth


In [2]:
!pip install --quiet unsloth
print('installed')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 130.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/lemmatizer_adapters'
os.makedirs(SAVE_DIR, exist_ok=True)
print('adapters will be saved to:', SAVE_DIR)


Mounted at /content/drive
adapters will be saved to: /content/drive/MyDrive/lemmatizer_adapters


## Upload the code and data

In [5]:
import zipfile, os
from google.colab import files

os.makedirs('/content/lemmatizer', exist_ok=True)
%cd /content/lemmatizer

up = files.upload()                     # pick colab_bundle.zip
for name in up:
    with zipfile.ZipFile(name) as z:
        z.extractall('/content/lemmatizer')
    print('extracted', name)

# Verify the four files arrived and the row counts are what we expect.
for p, want in (('train.py', None), ('skill.py', None),
                ('unsloth/data/train_reasoning.jsonl', 7200),
                ('unsloth/data/dev_reasoning.jsonl', 800)):
    n = sum(1 for _ in open(p, encoding='utf-8')) if want else None
    print('%-40s %s' % (p, ('%d rows' % n) if want else 'ok'))
    assert want is None or n == want, 'expected %d rows, got %d' % (want, n)


/content/lemmatizer


Saving colab_bundle.zip to colab_bundle.zip
extracted colab_bundle.zip
train.py                                 ok
skill.py                                 ok
unsloth/data/train_reasoning.jsonl       7200 rows
unsloth/data/dev_reasoning.jsonl         800 rows


In [6]:
!python train.py --model llama --bs 4 --accum 1 --max-steps 20 --evals 1


base      unsloth/Llama-3.2-3B-Instruct-bnb-4bit
template  llama-3.2
adapter   unsloth/lemmatizer_lora_llama
data      train_reasoning.jsonl / dev_reasoning.jsonl
effective batch 4  (4 x 1 accum)
seq len   896 | LoRA r16 on all 7 projections
budget    120 min

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprec

## Train Llama — the full epoch


In [7]:
!python train.py --model llama --bs 4 --accum 1 --budget-mins 0


base      unsloth/Llama-3.2-3B-Instruct-bnb-4bit
template  llama-3.2
adapter   unsloth/lemmatizer_lora_llama
data      train_reasoning.jsonl / dev_reasoning.jsonl
effective batch 4  (4 x 1 accum)
seq len   896 | LoRA r16 on all 7 projections
budget    none

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecate

## Save Llama to Drive

In [8]:
import shutil, os, json

def save(name):
    src = '/content/lemmatizer/unsloth/lemmatizer_lora_%s' % name
    dst = os.path.join(SAVE_DIR, 'lemmatizer_lora_%s' % name)
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(src, dst)
    # The loss curve lives in the checkpoints, not the adapter, and
    # plot_training.py reads it back on your laptop.
    runs = '/content/lemmatizer/unsloth/training_runs/lemmatizer_lora_%s' % name
    if os.path.isdir(runs):
        for ck in sorted(os.listdir(runs)):
            st = os.path.join(runs, ck, 'trainer_state.json')
            if os.path.exists(st):
                out = os.path.join(SAVE_DIR, 'training_runs',
                                   'lemmatizer_lora_%s' % name, ck)
                os.makedirs(out, exist_ok=True)
                shutil.copy2(st, out)
    print('saved ->', dst)
    print(open(os.path.join(dst, 'run_meta.json'), encoding='utf-8').read())

save('llama')


saved -> /content/drive/MyDrive/lemmatizer_adapters/lemmatizer_lora_llama
{
  "base": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
  "steps": 1800,
  "batch": 4,
  "accum": 1,
  "examples": 7200,
  "max_seq_len": 896,
  "lora_rank": 16,
  "best_eval_loss": 0.01695368066430092
}


## 8. Train Qwen

In [9]:
!python train.py --model qwen --bs 4 --accum 1 --budget-mins 0


base      unsloth/Qwen2.5-3B-Instruct-bnb-4bit
template  qwen-2.5
adapter   unsloth/lemmatizer_lora_qwen
data      train_reasoning.jsonl / dev_reasoning.jsonl
effective batch 4  (4 x 1 accum)
seq len   896 | LoRA r16 on all 7 projections
budget    none

[fairness] llama was trained for 1800 steps (7,200 examples, batch 4x1).
[fairness] This run has no --max-steps, so the probe will pick
           its own length from the clock and the two models will
           almost certainly differ. To compare pretraining rather than speed:
               --max-steps 1800

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = Fal

## 9. Save Qwen to Drive

In [10]:
save('qwen')


saved -> /content/drive/MyDrive/lemmatizer_adapters/lemmatizer_lora_qwen
{
  "base": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
  "steps": 1800,
  "batch": 4,
  "accum": 1,
  "examples": 7200,
  "max_seq_len": 896,
  "lora_rank": 16,
  "best_eval_loss": 0.015585736371576786
}
